# W261 Final Project Phase II - Baseline Modeling
**Team:** Section 3, Group 2

This notebook trains baseline models. We will first do model training and selection based on raw data. Then do the feature engineering. After that, we will use new feature set to go through the same modeling pipeline to do the model training and selection based on the feature engineered data. Eventually, we will do a final result comparison between models trained from raw data vs feature engineered data and get our final winner for baseline modeling part.

Going forward, we will use that model as our baseline model.

Load data from parquet

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

spark.conf.set("spark.sql.ansi.enabled", "false")

# Load data from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df_3m = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
df_12m = spark.read.parquet(f"{TEAM_DIR}/otpw_12m.parquet")

print(f"3-Month OTPW:  {df_3m.count():,} rows")
print(f"12-Month OTPW: {df_12m.count():,} rows")


# Data Cleaning & Feature Preparation

We structure our modeling in two phases:
1. **Baseline (raw features):** Train models using the original columns with minimal preprocessing (type casting, null imputation). This establishes a performance floor.
2. **Engineered features:** Apply feature transformations identified during EDA and evaluate improvement over baseline.

## Filter to FM-15 and Remove Cancelled Flights

We filter both datasets to FM-15 records only (87.5% of flights) to ensure consistent hourly weather coverage, and exclude cancelled flights since they have no departure delay to predict.

In [0]:
from pyspark.sql.functions import regexp_replace
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time

df_model_3m = df_3m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)
df_model_12m = df_12m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)

print(f"3-Month (FM-15, non-cancelled):  {df_model_3m.count():,} rows")
print(f"12-Month (FM-15, non-cancelled): {df_model_12m.count():,} rows")

### Clean Weather Columns

Raw weather values contain extraneous characters (e.g., `'s'` suffixes for suspect readings, `'*'` placeholders). We apply regex rules from our Phase I weather EDA to strip these and cast to proper numeric types.

In [0]:
hourly_clean_rules = {
    "HourlyStationPressure": ["s", "", "double"],
    "HourlySeaLevelPressure": ["s", "", "double"],
    "HourlyAltimeterSetting": ["s", "", "double"],
    "HourlyDewPointTemperature": ["[^0-9\\.\\-]", "", "double"],
    "HourlyDryBulbTemperature": ["[^0-9\\.\\-]", "", "double"],
    "HourlyWindSpeed": ["s", "", "double"],
    "HourlyWindGustSpeed": ["s", "", "double"],
    "HourlyVisibility": ["[^0-9\\.]", "", "double"],
    "HourlyRelativeHumidity": ["\\*", "0", "double"],
    "HourlyWindDirection": ["[^0-9]", "", "double"],
    "HourlyPrecipitation": ["[^0-9\\.]", "", "double"],
}

def clean_weather(df):
    for c_name, (regex_str, replace_with, cast_type) in hourly_clean_rules.items():
        if c_name in df.columns:
            df = df.withColumn(c_name, regexp_replace(col(c_name), regex_str, replace_with).cast(cast_type))
    return df

df_model_3m = clean_weather(df_model_3m)
df_model_12m = clean_weather(df_model_12m)
print("Weather columns cleaned and cast to numeric.")

### Select Baseline Features and Impute Nulls

We select only the raw features identified in the EDA with no transformations as this moment. Just type casting and null imputation. Per the NOAA documentation, FM-15 weather nulls are imputed as 0 since they represent "none observed." This gives us a clean baseline to measure raw feature predictive power before applying any feature engineering.

In [0]:
from pyspark.sql.functions import log1p, mean, stddev, when, concat_ws

label_col = "DEP_DEL15"

raw_cols = [
    label_col, "DISTANCE", "CRS_DEP_TIME", "FL_DATE",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH"
]

numeric_features = [
    "DISTANCE",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity"
]

categorical_features = ["OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR"]

def prepare_features_baseline(df):
    df = df.select(raw_cols)
    df = df.withColumn(label_col, col(label_col).cast("int"))
    df = df.filter(col(label_col).isNotNull())

    for c in numeric_features:
        df = df.withColumn(c, col(c).cast("double"))
        df = df.fillna({c: 0.0})

    # Extract hour of day from scheduled departure time
    df = df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))
    df = df.drop("CRS_DEP_TIME")

    return df

df_model_3m = prepare_features_baseline(df_model_3m)
df_model_12m = prepare_features_baseline(df_model_12m)

print(f"3-Month prepared:  {df_model_3m.count():,} rows x {len(df_model_3m.columns)} columns")
print(f"12-Month prepared: {df_model_12m.count():,} rows x {len(df_model_12m.columns)} columns")
df_model_12m.printSchema()

Following are all features we will go ahead using in our baseline modeling:

-  |-- DEP_DEL15: integer
-  |-- DISTANCE: double
-  |-- HourlyDryBulbTemperature: double
-  |-- HourlyDewPointTemperature: double
-  |-- HourlyWindSpeed: double 
-  |-- HourlyWindGustSpeed: double
-  |-- HourlyVisibility: double
-  |-- HourlyPrecipitation: double
-  |-- HourlyRelativeHumidity: double
-  |-- OP_UNIQUE_CARRIER: string
-  |-- ORIGIN: string
-  |-- DAY_OF_WEEK: string
-  |-- MONTH: string
-  |-- DEP_HOUR: integer

String features will be handled later in the baseline modeling part.

# Baseline Modeling (Raw Features)

We first train models using the raw features with no transformations. This establishes a performance baseline before applying feature engineering.

### Build Spark ML Pipeline

The pipeline applies `StringIndexer`, `OneHotEncoder` for each categorical feature.

In [0]:
cat_features_to_encode = categorical_features

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_features_to_encode
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in cat_features_to_encode
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in cat_features_to_encode]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="skip")

pipeline_stages = indexers + encoders + [assembler]

print(f"Pipeline stages: {len(pipeline_stages)}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(cat_features_to_encode)}): {cat_features_to_encode}")
print(f"Total assembler inputs: {len(assembler_inputs)}")

### Train/Test Split

**3-Month dataset (stepping stone):** We split months month 1 and 2 for training and month 3 for testing. This validates the pipeline before scaling to the full 12-month dataset.

**12-Month dataset (primary evaluation):** We use **rolling cross-validation on a quarterly basis**. Standard k-fold CV is inappropriate for time-series data because randomly shuffling temporal data would allow the model to train on future flights and predict past ones, leaking temporal information. Instead, we use a sliding window approach for the first two folds and a cumulative training set for the final fold:

| Fold | Train | Test |
|------|-------|------|
| 1 | Q1 (months 1 - 3) | Q2 (months 4 - 6) |
| 2 | Q2 (months 4 - 6) | Q3 (months 7 - 9) |
| 3 | Q1 - Q3 (months 1 - 9) | Q4 (months 10 - 12) |

Folds 1 - 2 use a single-quarter sliding window to test how well patterns from one quarter generalize to the next. Fold 3 uses all available training data (Q1â€“Q3) to predict the held-out Q4 test set, simulating real-world deployment.

In [0]:
# 3-Month: simple split for pipeline validation
train_3m = df_model_3m.filter(col("MONTH").cast("int") <= 2)
test_3m = df_model_3m.filter(col("MONTH").cast("int") > 2)

print("3-Month split (stepping stone):")
print(f"  Train (months 1-2): {train_3m.count():,}")
print(f"  Test  (month 3):    {test_3m.count():,}")


### Evaluation Metrics

For airlines, the cost of a **false negative** (predicting on-time when the flight is actually delayed) is significantly higher than a **false positive** (predicting delayed when the flight departs on time):

- **False Negative (missed delay):** The airline fails to prepare: crew scheduling is disrupted last-minute, gate reassignments cascade, passengers miss connections, and a single missed delay can trigger a chain of downstream disruptions across the network. The financial and reputational cost is high.
- **False Positive (false alarm):** The airline pre-allocates standby resources that weren't needed: extra crew on standby, a backup gate reserved, passengers notified of a potential delay. The cost is moderate (some wasted resources) but overall harmless.

This makes **recall** our primary metric for model selection: we want to maximize the fraction of actual delays we correctly identify, even at the cost of some false alarms. Precision is a secondary metric to ensure the model doesn't generate so many false alarms.

| Metric  | Why |
|--------|-----|
| **Recall** | Maximizes detection of actual delays, directly reduces the high-cost false negatives |
| **F2**  | Combines precision and recall into one score, while putting more weight on recall, which fits our goal of avoiding missed delays |
| **Precision** | Ensures predicted delays are actionable, not just noise |
| **AUC-PR**  | Captures the full precision-recall trade-off across thresholds |

### Baseline Modeling Approaches

We use **Logistic Regression** as the baseline. We then compare against tree-based models:

- **Random Forest:** Handles non-linear feature interactions without explicit engineering. Robust to outliers and noisy features.
- **Gradient Boosted Trees:** Sequentially builds trees to correct previous errors, typically achieving higher predictive performance than Random Forest. More prone to overfitting.

From the airline's perspective: if the priority is **maximizing delay detection** (recall), tree-based models typically outperform linear baselines because delays are driven by complex feature combinations.

In [0]:
def evaluate_model(predictions, label_col="DEP_DEL15"):
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")

    tp = predictions.filter((col("prediction") == 1) & (col(label_col) == 1)).count()
    fp = predictions.filter((col("prediction") == 1) & (col(label_col) == 0)).count()
    fn = predictions.filter((col("prediction") == 0) & (col(label_col) == 1)).count()

    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f2 = round(5 * precision * recall / (4 * precision + recall), 4) if (precision + recall) > 0 else 0.0

    return {
        "AUC-PR": round(evaluator_pr.evaluate(predictions), 4),
        "Precision": precision,
        "Recall": recall,
        "F2": f2,
    }

def run_experiment(model, pipeline_stages, train_df, test_df, model_name, dataset_name):
    full_pipeline = Pipeline(stages=pipeline_stages + [model])

    start = time.time()
    fitted = full_pipeline.fit(train_df)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_df)
    test_preds = fitted.transform(test_df)

    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    print(f"\n{'='*60}")
    print(f"{model_name} on {dataset_name}")
    print(f"Training time: {train_time}s")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"{'-'*42}")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Train Time (s)": train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }, fitted

## 3-Month Baseline Experiments

We start with the 3-month dataset to validate the pipeline before scaling to the full 12-month cross-validation. This uses raw features with undersampling for class imbalance.

In [0]:
results_3m = []

# --- Undersample majority class (on-time flights) to address class imbalance ---
def undersample_majority(df, label_col="DEP_DEL15", seed=42):
    """Undersample the majority class (label=0) to match the minority class count."""
    major_df = df.filter(col(label_col) == 0)
    minor_df = df.filter(col(label_col) == 1)
    major_count = major_df.count()
    minor_count = minor_df.count()
    undersample_ratio = minor_count / major_count
    major_undersampled = major_df.sample(withReplacement=False, fraction=undersample_ratio, seed=seed)
    return major_undersampled.unionAll(minor_df)

train_3m_balanced = undersample_majority(train_3m)
delayed_pct = train_3m_balanced.filter(col(label_col) == 1).count() / train_3m_balanced.count()
print(f"After undersampling - delayed class ratio: {delayed_pct:.2%}")
print(f"Balanced train size: {train_3m_balanced.count():,}")

lr_3m = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20)
lr_3m_result, lr_3m_fitted = run_experiment(lr_3m, pipeline_stages, train_3m_balanced, test_3m, "Logistic Regression", "3-Month")
results_3m.append(lr_3m_result)

In [0]:
rf_3m = RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42)
rf_3m_result, rf_3m_fitted = run_experiment(rf_3m, pipeline_stages, train_3m_balanced, test_3m, "Random Forest", "3-Month")
results_3m.append(rf_3m_result)

In [0]:
gbt_3m = GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42)
gbt_3m_result, gbt_3m_fitted = run_experiment(gbt_3m, pipeline_stages, train_3m_balanced, test_3m, "Gradient Boosted Trees", "3-Month")
results_3m.append(gbt_3m_result)

### 3-Month Baseline Results Summary

In [0]:
results_3m_df = pd.DataFrame(results_3m)
display(results_3m_df)


## 12-Month Baseline Cross Validation Experiments

We apply the 3-fold quarterly rolling CV described in the Train/Test Split section.

### Leakage Prevention

To ensure no future information leaks into training, we enforce the following safeguards in every fold:

1. **Temporal separation:** Each fold trains only on past or present quarters and tests on a future quarter. No random shuffling across time.
2. **Undersampling on train only:** Majority-class downsampling is applied exclusively to the training fold after the temporal split.
3. **Scaling on train only (FE section):** Z-score statistics (mean, std) are computed from the training fold and applied to both train and test. The test fold never influences scaling parameters.
4. **No target leakage:** We predict `DEP_DEL15` (binary delay indicator) using only features available before departure: weather observations, schedule metadata, and airport/carrier identifiers. No post-departure columns (actual delay minutes, arrival info) are included.

### Overfitting Detection

We report **both train and test F2** for every fold. A large gap (train >> test) signals overfitting, the model memorizes training data but fails to generalize. A small gap indicates healthy generalization.

In [0]:
# 3-fold quarterly rolling CV
cv_folds = [
    {"train_start": 1, "train_end": 3,  "test_start": 4,  "test_end": 6,  "label": "Train Q1, Test Q2"},
    {"train_start": 4, "train_end": 6,  "test_start": 7,  "test_end": 9,  "label": "Train Q2, Test Q3"},
    {"train_start": 1, "train_end": 9,  "test_start": 10, "test_end": 12, "label": "Train Q1-Q3, Test Q4"},
]

def rolling_cv(df, model_fn, pipeline_stages, folds, label_col="DEP_DEL15", scaling_fn=None):
    """Rolling cross-validation with quarterly folds and majority undersampling.
    Reports both train and test metrics per fold for overfitting analysis.
    If scaling_fn is provided, it is called as scaling_fn(train, test) per fold."""
    fold_results = []
    fitted_models = []

    for i, fold in enumerate(folds, 1):
        train_fold = df.filter(
            (col("MONTH").cast("int") >= fold["train_start"]) &
            (col("MONTH").cast("int") <= fold["train_end"])
        )
        test_fold = df.filter(
            (col("MONTH").cast("int") >= fold["test_start"]) &
            (col("MONTH").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        # Apply per-fold scaling if provided (fit on train only)
        if scaling_fn:
            train_fold, test_fold = scaling_fn(train_fold, test_fold)

        # Undersample majority class in the training fold
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        model = model_fn()
        full_pipeline = Pipeline(stages=pipeline_stages + [model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        fitted_models.append(fitted)

        # Evaluate on both train and test
        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  {'-'*38}")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    fold_df = pd.DataFrame(fold_results)
    return fold_df, fitted_models

print("rolling_cv function defined (3 quarterly folds, with train+test metrics).")

In [0]:
print("=" * 60)
print("Rolling CV: Logistic Regression (3 quarterly folds)")
print("=" * 60)
lr_cv, lr_cv_models = rolling_cv(
    df_model_12m,
    model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
    pipeline_stages=pipeline_stages, folds=cv_folds
)

print("\n" + "=" * 60)
print("Rolling CV: Random Forest (3 quarterly folds)")
print("=" * 60)
rf_cv, rf_cv_models = rolling_cv(
    df_model_12m,
    model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
    pipeline_stages=pipeline_stages, folds=cv_folds
)

# print("\n" + "=" * 60)
# print("Rolling CV: Gradient Boosted Trees (3 quarterly folds)")
# print("=" * 60)
# gbt_cv, gbt_cv_models = rolling_cv(
#     df_model_12m,
#     model_fn=lambda: GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
#     pipeline_stages=pipeline_stages, folds=cv_folds
# )

In [0]:
# Train vs Test across all metrics per fold — detect overfitting
models = ["Logistic Regression", "Random Forest"]
model_data = {"Logistic Regression": lr_cv, "Random Forest": rf_cv}
metrics = ["F2", "Recall", "Precision", "AUC-PR"]

fig, axes = plt.subplots(len(metrics), len(models), figsize=(6 * len(models), 4 * len(metrics)), sharey="row")

fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(models):
        ax = axes[row, col_idx]
        df = model_data[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Test", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        # Annotate gap
        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("Baseline Rolling CV: Train vs Test", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 12-Month Baseline Cross Validation Results

**Consistency across folds:**

- **Logistic Regression is unstable.** Fold 1 test F2 = 0.24, Fold 2 jumps to 0.51, Fold 3 = 0.49. The recall swings wildly (0.23 to 0.69 to 0.73), meaning LR's performance depends heavily on which quarter it trains on. It cannot reliably generalize across seasonal patterns.

- **Random Forest is more stable.** Test F2 stays in a tight band across all folds (0.44 to 0.49 to 0.46). Recall is similarly stable (0.50 to 0.63 to 0.63). This model generalizes reliably regardless of temporal shift.

**Overfitting analysis (train-test gap):**

- **LR overfits severely on Fold 1** (F2 gap = +0.42): train F2 = 0.65 but test F2 = 0.24. The model fits the training data well but completely fails on Q2. Folds 2 and 3 show smaller F2 gaps (+0.15, +0.17).

- **RF has moderate, stable overfitting** (F2 gaps: +0.19, +0.15, +0.19). The gap is consistent across folds.

- **Precision gap is large for both models (~0.33–0.41).** Train precision is ~0.63 (balanced training set), but test precision drops to ~0.21–0.30 on the imbalanced test set. This is expected: undersampling creates a 50/50 training distribution, so the model sees delays as common. At test time, delays are only ~20% of flights, so the model over-predicts delays. This is a consequence of undersampling.

**Conclusion:** **Random Forest is the stronger baseline.** It delivers consistent delay detection across all quarters and shows predictable, moderate overfitting. LR is unreliable, its Fold 1 collapse (F2 = 0.24) disqualifies it despite decent Fold 3 performance. The primary bottleneck for both models is low test precision, which we will address in Phase 3.

## Save and Load Baseline Models

Save all fitted baseline pipeline models to our team DBFS directory for reuse in Phase 3 without retraining.

In [0]:
# Save fitted models
model_dir = f"{TEAM_DIR}/models"

lr_3m_fitted.write().overwrite().save(f"{model_dir}/lr_3m")
rf_3m_fitted.write().overwrite().save(f"{model_dir}/rf_3m")
gbt_3m_fitted.write().overwrite().save(f"{model_dir}/gbt_3m")

# Save the fold 3 models (trained on Q1-Q3, the full training set)
lr_cv_models[2].write().overwrite().save(f"{model_dir}/lr_12m")
rf_cv_models[2].write().overwrite().save(f"{model_dir}/rf_12m")
# gbt_cv_models[2].write().overwrite().save(f"{model_dir}/gbt_12m")

print(f"All models saved to {model_dir}")

In [0]:
# Load models
from pyspark.ml import PipelineModel

lr_3m_loaded = PipelineModel.load(f"{model_dir}/lr_3m")
rf_3m_loaded = PipelineModel.load(f"{model_dir}/rf_3m")
gbt_3m_loaded = PipelineModel.load(f"{model_dir}/gbt_3m")

lr_12m_loaded = PipelineModel.load(f"{model_dir}/lr_12m")
rf_12m_loaded = PipelineModel.load(f"{model_dir}/rf_12m")
# gbt_12m_loaded = PipelineModel.load(f"{model_dir}/gbt_12m")

print("All models loaded.")

# Feature Engineering & Second Round of Training

Feature transformations were developed and validated in the [Phase II Feature Engineering notebook](Phase%20II%20Feature%20Engineering.ipynb). In this section, we apply those findings to our dataset and run the same modeling pipeline to compare engineered features against the raw baseline.

The following transformations are applied:

| Raw Feature | Transformation | Engineered Feature | Rationale |
|---|---|---|---|
| HourlyPrecipitation | Binary (> 0 then 1, else 0) | `PrecipBinary` | Heavily skewed; presence of rain matters more than amount |
| HourlyRelativeHumidity | Z-score normalization | `HourlyRelativeHumidity_scaled` | Approximately normal; standardized for model convergence |
| HourlyDryBulbTemperature | Z-score normalization | `HourlyDryBulbTemperature_scaled` | Approximately normal; standardized for model convergence |
| HourlyWindSpeed | Log-transform + Z-score | `HourlyWindSpeed_scaled` | Right-skewed; log normalizes the distribution |
| HourlyVisibility | Bucketed into 4 categories | `VisibilityCat` | Left-skewed with cap at 10; categorical bins capture operational impact |
| DAY_OF_WEEK | Binary (6,7 then 1, else 0) | `IsWeekend` | Weekend vs weekday delay pattern |
| ORIGIN + DEST | Concatenation | `Route` | Captures route-specific delay tendencies |

**Note:** Z-score statistics (mean, std) are computed on the **training set only** to avoid data leakage, then applied to both train and test sets.

In [0]:
# Re-load clean data (before baseline feature selection was applied)
df_fe_3m = df_3m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)
df_fe_12m = df_12m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)
df_fe_3m = clean_weather(df_fe_3m)
df_fe_12m = clean_weather(df_fe_12m)

# Columns to select from the raw data
fe_raw_cols = [
    label_col, "DISTANCE", "CRS_DEP_TIME", "FL_DATE",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DAY_OF_WEEK", "MONTH"
]

def prepare_features_engineered(df, extra_cols=None):
    """Apply feature engineering transformations.
    extra_cols: list of additional columns to keep through the select (e.g., time-based features)."""
    select_cols = fe_raw_cols + (extra_cols or [])
    df = df.select(select_cols)
    df = df.withColumn(label_col, col(label_col).cast("int"))
    df = df.filter(col(label_col).isNotNull())

    # Cast and impute numeric columns
    for c in ["DISTANCE", "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
              "HourlyWindSpeed", "HourlyWindGustSpeed", "HourlyVisibility",
              "HourlyPrecipitation", "HourlyRelativeHumidity"]:
        df = df.withColumn(c, col(c).cast("double"))
        df = df.fillna({c: 0.0})

    # DEP_HOUR from CRS_DEP_TIME
    df = df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))
    df = df.drop("CRS_DEP_TIME")

    # PrecipBinary: 1 if any precipitation, 0 otherwise
    df = df.withColumn("PrecipBinary", when(col("HourlyPrecipitation") > 0, 1.0).otherwise(0.0))

    # VisibilityCat: bucket into 4 categories (ordinal: 0=worst, 3=best)
    df = df.withColumn("VisibilityCat",
        when(col("HourlyVisibility") < 2, 0)
        .when((col("HourlyVisibility") >= 2) & (col("HourlyVisibility") < 5), 1)
        .when((col("HourlyVisibility") >= 5) & (col("HourlyVisibility") < 10), 2)
        .otherwise(3))

    # IsWeekend: 1 if Saturday(6) or Sunday(7)
    df = df.withColumn("IsWeekend", when(
        (col("DAY_OF_WEEK").cast("int") == 6) | (col("DAY_OF_WEEK").cast("int") == 7), 1
    ).otherwise(0))

    # Route: ORIGIN_DEST combination
    df = df.withColumn("Route", concat_ws("_", col("ORIGIN"), col("DEST")))

    # Log-transform wind speed (scaling deferred)
    df = df.withColumn("HourlyWindSpeed_log", log1p(col("HourlyWindSpeed")))

    return df

df_fe_3m = prepare_features_engineered(df_fe_3m)
df_fe_12m = prepare_features_engineered(df_fe_12m)

print(f"3-Month FE prepared:  {df_fe_3m.count():,} rows x {len(df_fe_3m.columns)} columns")
print(f"12-Month FE prepared: {df_fe_12m.count():,} rows x {len(df_fe_12m.columns)} columns")

### Scaling and Pipeline Setup

Z-score scaling for humidity, temperature, and wind speed must be fit on **training data only** to prevent data leakage. We wrap the scaling into the `rolling_cv_fe` function so each fold computes its own statistics from the training portion.

The engineered feature lists replace the baseline ones:
- **Numeric:** DISTANCE, HourlyDryBulbTemperature_scaled, HourlyDewPointTemperature, HourlyWindSpeed_scaled, HourlyWindGustSpeed, PrecipBinary, HourlyRelativeHumidity_scaled
- **Categorical:** OP_UNIQUE_CARRIER, ORIGIN, DAY_OF_WEEK, MONTH, DEP_HOUR, VisibilityCat, IsWeekend, Route

In [0]:
# Engineered feature lists
fe_numeric_features = [
    "DISTANCE",
    "HourlyDryBulbTemperature_scaled", "HourlyDewPointTemperature",
    "HourlyWindSpeed_scaled", "HourlyWindGustSpeed",
    "PrecipBinary",
    "HourlyRelativeHumidity_scaled",
    "VisibilityCat", "IsWeekend"
]

fe_categorical_features = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
    "Route"
]

def apply_scaling(train_df, test_df):
    """Compute z-score stats on train, apply to both train and test."""
    # Z-score for humidity and temperature
    for c in ["HourlyRelativeHumidity", "HourlyDryBulbTemperature"]:
        stats = train_df.select(mean(c).alias("m"), stddev(c).alias("s")).collect()[0]
        train_df = train_df.withColumn(f"{c}_scaled", (col(c) - stats["m"]) / stats["s"])
        test_df = test_df.withColumn(f"{c}_scaled", (col(c) - stats["m"]) / stats["s"])

    # Log-transform + z-score for wind speed
    stats_ws = train_df.select(
        mean("HourlyWindSpeed_log").alias("m"),
        stddev("HourlyWindSpeed_log").alias("s")
    ).collect()[0]
    train_df = train_df.withColumn("HourlyWindSpeed_scaled",
        (col("HourlyWindSpeed_log") - stats_ws["m"]) / stats_ws["s"])
    test_df = test_df.withColumn("HourlyWindSpeed_scaled",
        (col("HourlyWindSpeed_log") - stats_ws["m"]) / stats_ws["s"])

    # Fill any NaN from scaling
    for c in ["HourlyDryBulbTemperature_scaled", "HourlyRelativeHumidity_scaled", "HourlyWindSpeed_scaled"]:
        train_df = train_df.fillna({c: 0.0})
        test_df = test_df.fillna({c: 0.0})

    return train_df, test_df

# Build pipeline with engineered features
fe_cat_to_encode = fe_categorical_features

fe_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in fe_cat_to_encode
]
fe_encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in fe_cat_to_encode
]
fe_assembler_inputs = fe_numeric_features + [f"{c}_ohe" for c in fe_cat_to_encode]
fe_assembler = VectorAssembler(inputCols=fe_assembler_inputs, outputCol="features", handleInvalid="skip")
fe_pipeline_stages = fe_indexers + fe_encoders + [fe_assembler]

print(f"FE Pipeline stages: {len(fe_pipeline_stages)}")
print(f"FE Numeric features ({len(fe_numeric_features)}): {fe_numeric_features}")
print(f"FE Categorical features ({len(fe_cat_to_encode)}): {fe_cat_to_encode}")
print(f"FE Assembler inputs: {len(fe_assembler_inputs)}")

## 3-Month Feature Engineered Experiments

In [0]:
# 3-Month split for FE
train_3m_fe = df_fe_3m.filter(col("MONTH").cast("int") <= 2)
test_3m_fe = df_fe_3m.filter(col("MONTH").cast("int") > 2)

# Apply scaling (fit on train only)
train_3m_fe, test_3m_fe = apply_scaling(train_3m_fe, test_3m_fe)

# Undersample
train_3m_fe_balanced = undersample_majority(train_3m_fe)

print(f"3-Month FE split:")
print(f"  Train (months 1-2): {train_3m_fe.count():,} -> balanced: {train_3m_fe_balanced.count():,}")
print(f"  Test  (month 3):    {test_3m_fe.count():,}")

results_3m_fe = []

lr_3m_fe = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20)
lr_3m_fe_result, lr_3m_fe_fitted = run_experiment(lr_3m_fe, fe_pipeline_stages, train_3m_fe_balanced, test_3m_fe, "Logistic Regression", "3-Month FE")
results_3m_fe.append(lr_3m_fe_result)

rf_3m_fe = RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42)
rf_3m_fe_result, rf_3m_fe_fitted = run_experiment(rf_3m_fe, fe_pipeline_stages, train_3m_fe_balanced, test_3m_fe, "Random Forest", "3-Month FE")
results_3m_fe.append(rf_3m_fe_result)

# gbt_3m_fe = GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42)
# gbt_3m_fe_result, gbt_3m_fe_fitted = run_experiment(gbt_3m_fe, fe_pipeline_stages, train_3m_fe_balanced, test_3m_fe, "Gradient Boosted Trees", "3-Month FE")
# results_3m_fe.append(gbt_3m_fe_result)

results_3m_fe_df = pd.DataFrame(results_3m_fe)
display(results_3m_fe_df)

## 12-Month Feature Engineered Cross Validation

In [0]:
print("=" * 60)
print("FE Rolling CV: Logistic Regression")
print("=" * 60)
lr_fe_cv, lr_fe_cv_models = rolling_cv(
    df_fe_12m, model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
    pipeline_stages=fe_pipeline_stages, folds=cv_folds, scaling_fn=apply_scaling
)

print("\n" + "=" * 60)
print("FE Rolling CV: Random Forest")
print("=" * 60)
rf_fe_cv, rf_fe_cv_models = rolling_cv(
    df_fe_12m, model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
    pipeline_stages=fe_pipeline_stages, folds=cv_folds, scaling_fn=apply_scaling
)

# print("\n" + "=" * 60)
# print("FE Rolling CV: Gradient Boosted Trees")
# print("=" * 60)
# gbt_fe_cv, gbt_fe_cv_models = rolling_cv(
#     df_fe_12m, model_fn=lambda: GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
#     pipeline_stages=fe_pipeline_stages, folds=cv_folds, scaling_fn=apply_scaling
# )

### 12-Month FE Cross Validation Results

In [0]:
# Train vs Test across all metrics per fold — detect overfitting (FE)
# fe_models = ["Logistic Regression", "Random Forest", "Gradient Boosted Trees"]
fe_models = ["Logistic Regression", "Random Forest"]
fe_model_data = {
    "Logistic Regression": lr_fe_cv,
    "Random Forest": rf_fe_cv,
    # "Gradient Boosted Trees": gbt_fe_cv,
}
metrics = ["F2", "Recall", "Precision", "AUC-PR"]

fig, axes = plt.subplots(len(metrics), len(fe_models), figsize=(6 * len(fe_models), 4 * len(metrics)), sharey="row")

fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(fe_models):
        ax = axes[row, col_idx]
        df = fe_model_data[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Test", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(fe_models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("FE Rolling CV: Train vs Test", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Raw Features vs Engineered Features Comparison

We compare the 12-month CV results between raw baseline features and engineered features to determine whether the transformations improved delay detection.

In [0]:
# Fold 3 comparison: Raw vs Engineered (final holdout)
def extract_fold3(cv_df, model_name, feature_set):
    row = cv_df[cv_df["Fold"] == 3].iloc[0]
    return {
        "Model": model_name, "Features": feature_set,
        "Train_F2": row["Train_F2"], "Test_F2": row["Test_F2"],
        "Test_Recall": row["Test_Recall"], "Test_Precision": row["Test_Precision"],
        "Test_AUC-PR": row["Test_AUC-PR"],
    }

fold3_rows = [
    extract_fold3(lr_cv, "Logistic Regression", "Raw"),
    extract_fold3(rf_cv, "Random Forest", "Raw"),
    extract_fold3(lr_fe_cv, "Logistic Regression", "Engineered"),
    extract_fold3(rf_fe_cv, "Random Forest", "Engineered"),
    # extract_fold3(gbt_fe_cv, "Gradient Boosted Trees", "Engineered"),
]

comparison = pd.DataFrame(fold3_rows)
comparison["Overfit_Gap"] = (comparison["Train_F2"] - comparison["Test_F2"]).round(4)

print("=" * 70)
print("Fold 3 Final Holdout: Raw vs Engineered")
print("=" * 70)
display(comparison.round(4))

best = comparison.iloc[0]
print(
    f"\nBest final holdout by Test F2: {best['Model']} ({best['Features']}) "
    f"with Test F2 = {best['Test_F2']:.4f}, Overfit Gap = {best['Overfit_Gap']:+.4f}"
)

# Time-Based Features (RFM-Style)

To capture historical delay patterns, we create time-based features inspired by the RFM (Recency, Frequency, Monetary) framework:

| Feature | Type | Description |
|---------|------|-------------|
| `origin_delay_freq_7d` | Frequency | Fraction of flights delayed at this origin in the past 7 days
| `carrier_delay_freq_7d` | Frequency | Fraction of flights delayed by this carrier in the past 7 days
| `origin_days_since_delay` | Recency | Days since the most recent delay at this origin airport 

**Why these features matter:** Our baseline models use only point-in-time weather and static schedule metadata. They have no sense of *recent airport or carrier operational stress*. The Frequency features capture the rolling delay rate; the Recency feature captures whether disruption is *active right now* (value = 1) vs. *days in the past* (value = 5+). An airport with 20% weekly delay rate but no delays in the last 3 days is different from one that had delays yesterday.

**Will these features have leakage issue?** No, all features use Spark window functions ordered by `FL_DATE_days` with strict backward-looking ranges (`rangeBetween(-7, -1)` for frequency, unbounded past for recency). Each flight sees only delays from **previous days**, never from the same day or future.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, datediff, avg as spark_avg, max as spark_max

def add_time_features(df):
    """Add RFM-style time-based features using backward-looking windows only."""
    df = df.withColumn("FL_DATE_dt", to_date(col("FL_DATE")))
    df = df.withColumn("FL_DATE_days", datediff(col("FL_DATE_dt"), lit("2015-01-01")))

    # --- Frequency: fraction of flights delayed in past 7 days ---
    origin_freq_window = Window.partitionBy("ORIGIN") \
        .orderBy("FL_DATE_days") \
        .rangeBetween(-7, -1)
    df = df.withColumn("origin_delay_freq_7d",
        spark_avg(col(label_col).cast("double")).over(origin_freq_window))

    carrier_freq_window = Window.partitionBy("OP_UNIQUE_CARRIER") \
        .orderBy("FL_DATE_days") \
        .rangeBetween(-7, -1)
    df = df.withColumn("carrier_delay_freq_7d",
        spark_avg(col(label_col).cast("double")).over(carrier_freq_window))

    # --- Recency: days since the most recent delay at this origin ---
    # For each row, find the max FL_DATE_days among all prior rows at the same origin where DEP_DEL15=1
    # We create a helper column that is FL_DATE_days only when delayed, else null
    df = df.withColumn("_delayed_day",
        when(col(label_col) == 1, col("FL_DATE_days")).otherwise(None))

    origin_recency_window = Window.partitionBy("ORIGIN") \
        .orderBy("FL_DATE_days") \
        .rowsBetween(Window.unboundedPreceding, -1)
    df = df.withColumn("_last_delay_day",
        spark_max("_delayed_day").over(origin_recency_window))
    df = df.withColumn("origin_days_since_delay",
        col("FL_DATE_days") - col("_last_delay_day"))

    # Fill nulls: no prior delay observed → use a large default (30 days = "no recent stress")
    df = df.fillna({
        "origin_delay_freq_7d": 0.0,
        "carrier_delay_freq_7d": 0.0,
        "origin_days_since_delay": 30,
    })

    # Drop helper columns
    df = df.drop("FL_DATE_dt", "FL_DATE_days", "_delayed_day", "_last_delay_day")

    return df

# Reload, add time features, then reuse prepare_features_engineered
df_rfm_12m = df_12m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)
df_rfm_12m = clean_weather(df_rfm_12m)
df_rfm_12m = add_time_features(df_rfm_12m)

rfm_extra_cols = ["origin_delay_freq_7d", "carrier_delay_freq_7d", "origin_days_since_delay"]
df_rfm_12m = prepare_features_engineered(df_rfm_12m, extra_cols=rfm_extra_cols)

# Cache to avoid recomputing window functions on every fold
df_rfm_12m = df_rfm_12m.cache()

print(f"12-Month RFM prepared: {df_rfm_12m.count():,} rows x {len(df_rfm_12m.columns)} columns")
print(f"Sample time features (from month 2+):")
df_rfm_12m.filter(col("MONTH").cast("int") >= 2) \
    .select("FL_DATE", "ORIGIN", "origin_delay_freq_7d", "carrier_delay_freq_7d", "origin_days_since_delay") \
    .dropDuplicates(["FL_DATE", "ORIGIN"]) \
    .orderBy("ORIGIN", "FL_DATE") \
    .show(10, truncate=False)

In [0]:
# RFM feature lists: add time-based features to existing FE features
rfm_numeric_features = fe_numeric_features + [
    "origin_delay_freq_7d", "carrier_delay_freq_7d", "origin_days_since_delay"
]
rfm_categorical_features = fe_categorical_features

# Build pipeline with RFM features
rfm_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in rfm_categorical_features
]
rfm_encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in rfm_categorical_features
]
rfm_assembler_inputs = rfm_numeric_features + [f"{c}_ohe" for c in rfm_categorical_features]
rfm_assembler = VectorAssembler(inputCols=rfm_assembler_inputs, outputCol="features", handleInvalid="skip")
rfm_pipeline_stages = rfm_indexers + rfm_encoders + [rfm_assembler]

print(f"RFM Pipeline stages: {len(rfm_pipeline_stages)}")
print(f"RFM Numeric features ({len(rfm_numeric_features)}): {rfm_numeric_features}")
print(f"Time-based features: origin_delay_freq_7d (F), carrier_delay_freq_7d (F), origin_days_since_delay (R)")

# Run rolling CV with RFM features
print("\n" + "=" * 60)
print("RFM Rolling CV: Logistic Regression")
print("=" * 60)
lr_rfm_cv, lr_rfm_models = rolling_cv(
    df_rfm_12m,
    model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
    pipeline_stages=rfm_pipeline_stages, folds=cv_folds, scaling_fn=apply_scaling
)

print("\n" + "=" * 60)
print("RFM Rolling CV: Random Forest")
print("=" * 60)
rf_rfm_cv, rf_rfm_models = rolling_cv(
    df_rfm_12m,
    model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
    pipeline_stages=rfm_pipeline_stages, folds=cv_folds, scaling_fn=apply_scaling
)

In [0]:
# Train vs Test across all metrics per fold
rfm_models = ["Logistic Regression", "Random Forest"]
rfm_model_data = {
    "Logistic Regression": lr_rfm_cv,
    "Random Forest": rf_rfm_cv,
}
metrics = ["F2", "Recall", "Precision", "AUC-PR"]

fig, axes = plt.subplots(len(metrics), len(rfm_models), figsize=(6 * len(rfm_models), 4 * len(metrics)), sharey="row")

fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(rfm_models):
        ax = axes[row, col_idx]
        df = rfm_model_data[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Test", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(rfm_models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("RFM Rolling CV: Train vs Test", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# Final Holdout Comparison: Raw vs FE vs RFM (Fold 3 only)
fold3_rows = [
    extract_fold3(lr_cv, "Logistic Regression", "Raw"),
    extract_fold3(rf_cv, "Random Forest", "Raw"),
    extract_fold3(lr_fe_cv, "Logistic Regression", "Engineered"),
    extract_fold3(rf_fe_cv, "Random Forest", "Engineered"),
    extract_fold3(lr_rfm_cv, "Logistic Regression", "RFM"),
    extract_fold3(rf_rfm_cv, "Random Forest", "RFM"),
]

comparison = pd.DataFrame(fold3_rows)
comparison["Overfit_Gap"] = (comparison["Train_F2"] - comparison["Test_F2"]).round(4)

print("=" * 70)
print("Fold 3 Final Holdout: Raw vs Engineered vs RFM")
print("=" * 70)
display(comparison.round(4))

best = comparison.iloc[0]
print(
    f"\nBest final holdout by Test F2: {best['Model']} ({best['Features']}) "
    f"with Test F2 = {best['Test_F2']:.4f}, Overfit Gap F2 (Train - Test) = {best['Overfit_Gap']:+.4f}"
)

# Hyperparameter Tuning via Grid Search

From the comparison table above, we select **Random Forest** as the model to tune for two reasons:

1. **Consistency:** RF delivers stable F2 across all three rolling CV folds regardless of feature set, unlike LR which collapses on Fold 1 (F2 = 0.24 for Raw, 0.26 for FE, 0.33 for RFM). A model that fails unpredictably on certain quarters cannot be trusted in deployment.

2. **Precision-recall trade-off:** Although LR (Raw) achieved the highest single-fold F2 (0.49 on Fold 3), RF shows a better AUC-PR trajectory with RFM features (0.27), meaning considering both precision and recall, it is a better choice.

We tune on the **RFM feature set** because:
- RFM features improved **AUC-PR** over Raw and FE for both models, indicating the time-based features add ranking quality even though the default threshold didn't translate to higher F2.
- The RFM features represent the richest feature set we've built, if hyperparameter tuning can improve RF's ability to exploit the recency and frequency signals, the combined effect of better features + better hyperparameters is the strongest candidate for Phase 3.

We search over `numTrees` and `maxDepth` using Fold 3 (train Q1–Q3, test Q4) as the evaluation split

In [0]:
# Grid search: RF on RFM features, Fold 3 (train Q1-Q3, test Q4)
fold3 = cv_folds[2]

train_gs = df_rfm_12m.filter(
    (col("MONTH").cast("int") >= fold3["train_start"]) &
    (col("MONTH").cast("int") <= fold3["train_end"])
)
test_gs = df_rfm_12m.filter(
    (col("MONTH").cast("int") >= fold3["test_start"]) &
    (col("MONTH").cast("int") <= fold3["test_end"])
)

# Apply scaling (fit on train only)
train_gs, test_gs = apply_scaling(train_gs, test_gs)

# Undersample
train_gs_balanced = undersample_majority(train_gs, seed=45)

print(f"Grid search data: train={train_gs_balanced.count():,}, test={test_gs.count():,}")

# Grid search
param_grid = [
    {"numTrees": 50,  "maxDepth": 5},
    {"numTrees": 50,  "maxDepth": 10},
    {"numTrees": 100, "maxDepth": 5},
    {"numTrees": 100, "maxDepth": 10},
    {"numTrees": 100, "maxDepth": 15},
    {"numTrees": 200, "maxDepth": 10},
    {"numTrees": 200, "maxDepth": 15},
]

gs_results = []
for params in param_grid:
    rf_gs = RandomForestClassifier(
        featuresCol="features", labelCol=label_col,
        numTrees=params["numTrees"], maxDepth=params["maxDepth"], seed=42
    )
    full_pipeline = Pipeline(stages=rfm_pipeline_stages + [rf_gs])

    start = time.time()
    fitted = full_pipeline.fit(train_gs_balanced)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_gs_balanced)
    test_preds = fitted.transform(test_gs)
    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    gs_results.append({
        "numTrees": params["numTrees"],
        "maxDepth": params["maxDepth"],
        "Train_F2": train_metrics["F2"],
        "Test_F2": test_metrics["F2"],
        "Test_Recall": test_metrics["Recall"],
        "Test_Precision": test_metrics["Precision"],
        "Test_AUC-PR": test_metrics["AUC-PR"],
        "Overfit_Gap": round(train_metrics["F2"] - test_metrics["F2"], 4),
        "Train_Time_s": train_time,
    })
    print(
        f"numTrees={params['numTrees']:>3}, maxDepth={params['maxDepth']:>2} | "
        f"Train F2={train_metrics['F2']:.4f}, Test F2={test_metrics['F2']:.4f} "
        f"(gap={train_metrics['F2'] - test_metrics['F2']:+.4f}) | {train_time}s"
    )

gs_df = pd.DataFrame(gs_results).sort_values("Test_F2", ascending=False).reset_index(drop=True)
print("\nGrid Search Results (sorted by Test F2):")
display(gs_df.round(4))

best_params = gs_df.iloc[0]
print(
    f"\nBest: numTrees={int(best_params['numTrees'])}, maxDepth={int(best_params['maxDepth'])} "
    f"with Test F2={best_params['Test_F2']:.4f}, "
    f"Overfit Gap={best_params['Overfit_Gap']:+.4f}"
)